In [3]:
import os
from dotenv import load_dotenv

load_dotenv('api.env')
OPENAQ_API_KEY=os.getenv('OPENAQ_API_KEY')

def find_locations_by_coords(city_name, lat, lon):
    url = "https://api.openaq.org/v3/locations"
    headers = {"X-API-Key": OPENAQ_API_KEY}
    params = {
        "coordinates": f"{lat},{lon}",
        "radius": 25000,  # 25km radius
        "country": "IN",
        "limit": 5
    }
    response = requests.get(url, headers=headers, params=params)
    data = response.json()
    print(f"\n{city_name}:")
    if not data.get('results'):
        print("No results found")
        return
    for loc in data['results']:
        print(f"ID: {loc['id']} | Name: {loc['name']} | Sensors: {len(loc.get('sensors',[]))}")
    print("---")

COORDS = {
    'Delhi':     (28.6139, 77.2090),
    'Mumbai':    (19.0760, 72.8777),
    'Chennai':   (13.0827, 80.2707),
    'Kolkata':   (22.5726, 88.3639),
    'Bangalore': (12.9716, 77.5946),
    'Hyderabad': (17.3850, 78.4867),
    'Pune':      (18.5204, 73.8567),
    'Ahmedabad': (23.0225, 72.5714),
    'Jaipur':    (26.9124, 75.7873),
    'Lucknow':   (26.8467, 80.9462),
}

for city, (lat, lon) in COORDS.items():
    find_locations_by_coords(city, lat, lon)


Delhi:
ID: 13 | Name: Delhi Technological University, Delhi - CPCB | Sensors: 3
ID: 15 | Name: IGI Airport | Sensors: 5
ID: 16 | Name: Civil Lines | Sensors: 4
ID: 17 | Name: R K Puram, Delhi - DPCC | Sensors: 18
ID: 50 | Name: Punjabi Bagh, Delhi - DPCC | Sensors: 18
---

Mumbai:
ID: 2598 | Name: Navi Mumbai Municipal Corporation Airoli | Sensors: 4
ID: 5593 | Name: Pimpleshwar Mandir, Thane - MPCB | Sensors: 6
ID: 6927 | Name: Colaba, Mumbai - MPCB | Sensors: 18
ID: 6943 | Name: Mahape, Navi Mumbai - MPCB | Sensors: 18
ID: 6945 | Name: Kurla, Mumbai - MPCB | Sensors: 18
---

Chennai:
ID: 378 | Name: Alandur Bus Depot | Sensors: 5
ID: 2461 | Name: US Diplomatic Post: Chennai | Sensors: 1
ID: 2549 | Name: IIT | Sensors: 5
ID: 2586 | Name: Manali, Chennai - CPCB | Sensors: 18
ID: 5655 | Name: Velachery Res. Area, Chennai - CPCB | Sensors: 18
---

Kolkata:
ID: 716 | Name: Rabindra Bharati University, Kolkata - WBSPCB | Sensors: 6
ID: 910 | Name: Victoria Memorial - WBSPCB | Sensors: 6
I

In [4]:
import os
from dotenv import load_dotenv

load_dotenv('api.env')
OPENAQ_API_KEY=os.getenv('OPENAQ_API_KEY')

def check_sensors(city_name, location_id):
    url = f"https://api.openaq.org/v3/locations/{location_id}"
    headers = {"X-API-Key": OPENAQ_API_KEY}
    response = requests.get(url, headers=headers)
    data = response.json()
    
    print(f"\n{city_name} (ID: {location_id}):")
    for sensor in data['results'][0]['sensors']:
        print(f"  {sensor['parameter']['name']} — {sensor['parameter']['units']}")

LOCATION_IDS = {
    'Delhi': 17, 'Mumbai': 6927, 'Chennai': 2586,
    'Kolkata': 5614, 'Bangalore': 594, 'Hyderabad': 407,
    'Pune': 5661, 'Ahmedabad': 5631, 'Jaipur': 5612, 'Lucknow': 2456
}

for city, loc_id in LOCATION_IDS.items():
    check_sensors(city, loc_id)


Delhi (ID: 17):
  co — µg/m³
  co — ppb
  no — ppb
  no2 — µg/m³
  no2 — ppb
  nox — ppb
  o3 — µg/m³
  o3 — µg/m³
  pm10 — µg/m³
  pm10 — µg/m³
  pm25 — µg/m³
  pm25 — µg/m³
  relativehumidity — %
  so2 — µg/m³
  so2 — ppb
  temperature — c
  wind_direction — deg
  wind_speed — m/s

Mumbai (ID: 6927):
  co — µg/m³
  co — ppb
  no — ppb
  no2 — µg/m³
  no2 — ppb
  nox — ppb
  o3 — µg/m³
  o3 — µg/m³
  pm10 — µg/m³
  pm10 — µg/m³
  pm25 — µg/m³
  pm25 — µg/m³
  relativehumidity — %
  so2 — ppb
  so2 — µg/m³
  temperature — c
  wind_direction — deg
  wind_speed — m/s

Chennai (ID: 2586):
  co — µg/m³
  co — ppb
  no — ppb
  no2 — µg/m³
  no2 — ppb
  nox — ppb
  o3 — µg/m³
  o3 — µg/m³
  pm10 — µg/m³
  pm10 — µg/m³
  pm25 — µg/m³
  pm25 — µg/m³
  relativehumidity — %
  so2 — ppb
  so2 — µg/m³
  temperature — c
  wind_direction — deg
  wind_speed — m/s

Kolkata (ID: 5614):
  co — ppb
  co — µg/m³
  no — ppb
  no2 — ppb
  no2 — µg/m³
  nox — ppb
  o3 — µg/m³
  o3 — µg/m³
  pm10 — µg/m³
  p

In [10]:
import os
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from dotenv import load_dotenv


load_dotenv('api.env') 
OPENAQ_API_KEY = os.getenv('OPENAQ_API_KEY')

if not OPENAQ_API_KEY:
    raise ValueError("API Key not found! Please check your api.env file.")

headers = {"X-API-Key": OPENAQ_API_KEY}     


CITIES = {
    'Delhi': {'lat':28.6139, 'lon': 77.2090},
    'Mumbai': {'lat':19.0760, 'lon': 72.8777},
    'Chennai': {'lat':13.0827, 'lon': 80.2707},
    'Kolkata': {'lat':22.5726, 'lon': 88.3639},
    'Bangalore': {'lat':12.9716, 'lon': 77.5946},
    'Hyderabad': {'lat': 17.3850, 'lon': 78.4867},
    'Pune': {'lat': 18.5204, 'lon': 73.8567},
    'Ahmedabad': {'lat': 23.0225, 'lon': 72.5714},
    'Jaipur': {'lat': 26.9124, 'lon': 75.7873},
    'Lucknow': {'lat': 26.8467, 'lon': 80.9462},
}

EXPECTED_POLLUTANTS = ['pm25', 'pm10', 'no', 'no2', 'nh3', 'co', 'so2', 'o3']


date_to = datetime.now()
date_from = date_to - timedelta(days=1825) 
fmt_date_to = date_to.strftime("%Y-%m-%dT%H:%M:%S%z") + "Z"
fmt_date_from = date_from.strftime("%Y-%m-%dT%H:%M:%S%z") + "Z"

all_city_data = []


print("Starting OpenAQ v3 Ground Sensor Data Extraction...")

for city, coords in CITIES.items():
    print(f"\nScanning {city} for physical stations...")
    
    loc_url = "https://api.openaq.org/v3/locations"
    loc_params = {
        "coordinates": f"{coords['lat']},{coords['lon']}",
        "radius": 25000, 
        "limit": 100 
    }
    
    try:
        loc_resp = requests.get(loc_url, headers=headers, params=loc_params)
        if loc_resp.status_code != 200:
            print(f"Could not fetch locations for {city}. Code: {loc_resp.status_code}")
            continue
            
        locations = loc_resp.json().get('results', [])
        if not locations:
            print(f"No physical stations found in {city} within 25km.")
            continue
            
        print(f"   Found {len(locations)} stations. Fetching 5 years of sensor data...")
        
        city_records = []
        
        for loc in locations:
            sensors = loc.get('sensors', [])
            for sensor in sensors:
                sensor_id = sensor.get('id')
                parameter = sensor.get('parameter', {}).get('name')
                
                if parameter not in EXPECTED_POLLUTANTS:
                    continue 
                    
                meas_url = f"https://api.openaq.org/v3/sensors/{sensor_id}/days"
                meas_params = {
                    "datetime_from": fmt_date_from,
                    "datetime_to": fmt_date_to,
                    "limit": 1000 
                }
                
                page = 1
                while True:
                    meas_params['page'] = page
                    meas_resp = requests.get(meas_url, headers=headers, params=meas_params)
                    
                    if meas_resp.status_code == 429:
                        print("Rate limit hit. Cooling down for 5 seconds...")
                        time.sleep(5)
                        continue 
                        
                    if meas_resp.status_code != 200:
                        break 
                        
                    meas_data = meas_resp.json().get('results', [])
                    if not meas_data:
                        break
                        
                    for row in meas_data:
                        city_records.append({
                            'Date': pd.to_datetime(row['period']['datetimeFrom']['utc']).date(),
                            'Parameter': parameter,
                            'Value': row.get('value', None)
                        })
                    
                    if len(meas_data) < 1000:
                        break
                    
                    page += 1
                    time.sleep(0.5) 
                    
            time.sleep(0.2) 

       
        if city_records:
            df_temp = pd.DataFrame(city_records)
            df_daily = df_temp.groupby(['Date', 'Parameter'])['Value'].mean().reset_index()
            df_pivot = df_daily.pivot(index='Date', columns='Parameter', values='Value')
            df_pivot = df_pivot.reindex(columns=EXPECTED_POLLUTANTS)
            
            
            missing_count = df_pivot.isnull().sum(axis=1)
            df_pivot = df_pivot[missing_count <= 4]
            
            
            df_pivot = df_pivot.interpolate(method='linear', limit_direction='both')
            
            df_pivot['City'] = city
            df_pivot.reset_index(inplace=True)
            all_city_data.append(df_pivot)
            print(f"Extracted {len(df_pivot)} safe days of data for {city}")

    except Exception as e:
        print(f"Error processing {city}: {e}")


if len(all_city_data) == 0:
    print("\nCRITICAL FAILURE: No data was collected.")
else:
    print("\nAssembling and cleaning the final dataset...")
    final_df = pd.concat(all_city_data, ignore_index=True)
    
    final_df.rename(columns={
        'pm25': 'PM2.5', 'pm10': 'PM10', 'no': 'NO', 'no2': 'NO2',
        'nh3': 'NH3', 'co': 'CO', 'so2': 'SO2', 'o3': 'O3'
    }, inplace=True)

    final_df['Date'] = pd.to_datetime(final_df['Date'])
    final_df['Year'] = final_df['Date'].dt.year
    final_df['Month'] = final_df['Date'].dt.month
    final_df['Seasons'] = final_df['Month'].map({
        12:'Winter', 1:'Winter', 2:'Winter',
        3:'Spring', 4:'Spring', 5:'Spring',
        6:'Monsoon', 7:'Monsoon', 8:'Monsoon', 9:'Monsoon',
        10:'Post-Monsoon', 11:'Post-Monsoon'
    })


    final_df.to_csv('openaq_ground_sensor_data.csv', index=False)
    print("Done! Saved successfully as 'openaq_ground_sensor_data.csv'")


Starting OpenAQ v3 Ground Sensor Data Extraction...

Scanning Delhi for physical stations...
   Found 94 stations. Fetching 5 years of sensor data...
Rate limit hit. Cooling down for 5 seconds...
Extracted 2406 safe days of data for Delhi

Scanning Mumbai for physical stations...
   Found 44 stations. Fetching 5 years of sensor data...
Extracted 1854 safe days of data for Mumbai

Scanning Chennai for physical stations...
   Found 11 stations. Fetching 5 years of sensor data...
Extracted 2195 safe days of data for Chennai

Scanning Kolkata for physical stations...
   Found 19 stations. Fetching 5 years of sensor data...
Extracted 2100 safe days of data for Kolkata

Scanning Bangalore for physical stations...
   Found 25 stations. Fetching 5 years of sensor data...
Error processing Bangalore: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

Scanning Hyderabad for physical stations...
   Found 14 stat

In [11]:
import numpy as np
import pandas as pd

df = pd.read_csv("openaq_ground_sensor_data.csv")
print(df.shape)
print(df.head())

(18740, 13)
         Date   PM2.5        PM10        NO         NO2  NH3      CO  \
0  2016-02-04  191.00  460.250000  30.75913  138.750000  NaN  4925.0   
1  2016-02-05  214.36  460.500000  30.75913   87.225000  NaN  2463.0   
2  2016-02-06  160.25  252.333333  30.75913   78.433333  NaN  1195.0   
3  2016-02-07  112.46  256.500000  30.75913   66.175000  NaN  1712.5   
4  2016-02-08  192.60  304.000000  30.75913   79.750000  NaN  1467.5   

         SO2         O3   City  Year  Month Seasons  
0  35.800000  13.265000  Delhi  2016      2  Winter  
1  35.775000  50.350000  Delhi  2016      2  Winter  
2  23.533333  31.233333  Delhi  2016      2  Winter  
3  23.375000  36.625000  Delhi  2016      2  Winter  
4  29.025000  47.575000  Delhi  2016      2  Winter  


In [12]:
df['City'].unique()

array(['Delhi', 'Mumbai', 'Chennai', 'Kolkata', 'Hyderabad', 'Pune',
       'Ahmedabad', 'Jaipur', 'Lucknow'], dtype=object)

In [13]:
import os
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv('api.env') 
OPENAQ_API_KEY = os.getenv('OPENAQ_API_KEY')
headers = {"X-API-Key": OPENAQ_API_KEY}     


CITIES = {'Bangalore': {'lat':12.9716, 'lon': 77.5946}}
EXPECTED_POLLUTANTS = ['pm25', 'pm10', 'no', 'no2', 'nh3', 'co', 'so2', 'o3']

date_to = datetime.now()
date_from = date_to - timedelta(days=1825) 
fmt_date_to = date_to.strftime("%Y-%m-%dT%H:%M:%S%z") + "Z"
fmt_date_from = date_from.strftime("%Y-%m-%dT%H:%M:%S%z") + "Z"

all_city_data = []


print("Fetching missing data for Bangalore...")
for city, coords in CITIES.items():
    loc_url = "https://api.openaq.org/v3/locations"
    loc_params = {"coordinates": f"{coords['lat']},{coords['lon']}", "radius": 25000, "limit": 100}
    
    try:
        loc_resp = requests.get(loc_url, headers=headers, params=loc_params)
        locations = loc_resp.json().get('results', [])
        print(f"   Found {len(locations)} stations. Downloading...")
        
        city_records = []
        for loc in locations:
            sensors = loc.get('sensors', [])
            for sensor in sensors:
                sensor_id = sensor.get('id')
                parameter = sensor.get('parameter', {}).get('name')
                
                if parameter not in EXPECTED_POLLUTANTS: continue 
                    
                meas_url = f"https://api.openaq.org/v3/sensors/{sensor_id}/days"
                meas_params = {"datetime_from": fmt_date_from, "datetime_to": fmt_date_to, "limit": 1000}
                
                page = 1
                while True:
                    meas_params['page'] = page
                    meas_resp = requests.get(meas_url, headers=headers, params=meas_params)
                    
                    if meas_resp.status_code == 429:
                        time.sleep(5)
                        continue 
                    if meas_resp.status_code != 200: break 
                        
                    meas_data = meas_resp.json().get('results', [])
                    if not meas_data: break
                        
                    for row in meas_data:
                        city_records.append({
                            'Date': pd.to_datetime(row['period']['datetimeFrom']['utc']).date(),
                            'Parameter': parameter,
                            'Value': row.get('value', None)
                        })
                    if len(meas_data) < 1000: break
                    page += 1
                    time.sleep(0.5) 
            time.sleep(0.2) 

        if city_records:
            df_temp = pd.DataFrame(city_records)
            df_daily = df_temp.groupby(['Date', 'Parameter'])['Value'].mean().reset_index()
            df_pivot = df_daily.pivot(index='Date', columns='Parameter', values='Value')
            df_pivot = df_pivot.reindex(columns=EXPECTED_POLLUTANTS)
            
            missing_count = df_pivot.isnull().sum(axis=1)
            df_pivot = df_pivot[missing_count <= 4]
            df_pivot = df_pivot.interpolate(method='linear', limit_direction='both')
            df_pivot['City'] = city
            df_pivot.reset_index(inplace=True)
            all_city_data.append(df_pivot)
            print(f"Extracted {len(df_pivot)} safe days for Bangalore")

    except Exception as e:
        print(f"Error: {e}")

if len(all_city_data) > 0:
    print("\nFormatting Bangalore data...")
    bng_df = pd.concat(all_city_data, ignore_index=True)
    bng_df.rename(columns={'pm25': 'PM2.5', 'pm10': 'PM10', 'no': 'NO', 'no2': 'NO2', 'nh3': 'NH3', 'co': 'CO', 'so2': 'SO2', 'o3': 'O3'}, inplace=True)
    bng_df['Date'] = pd.to_datetime(bng_df['Date'])
    bng_df['Year'] = bng_df['Date'].dt.year
    bng_df['Month'] = bng_df['Date'].dt.month
    bng_df['Seasons'] = bng_df['Month'].map({
        12:'Winter', 1:'Winter', 2:'Winter', 3:'Spring', 4:'Spring', 5:'Spring',
        6:'Monsoon', 7:'Monsoon', 8:'Monsoon', 9:'Monsoon', 10:'Post-Monsoon', 11:'Post-Monsoon'
    })

    existing_df = pd.read_csv('openaq_ground_sensor_data.csv') 
    
    complete_df = pd.concat([existing_df, bng_df], ignore_index=True)
    complete_df.to_csv('openaq_ground_sensor_data_complete.csv', index=False)
    print("SUCCESS! Saved as 'openaq_ground_sensor_data_complete.csv'")
else:
    print("Failed to get Bangalore data. Try running this patch cell again.")
  

Fetching missing data for Bangalore...
   Found 25 stations. Downloading...
Extracted 2234 safe days for Bangalore

Formatting Bangalore data...
SUCCESS! Saved as 'openaq_ground_sensor_data_complete.csv'


In [15]:
df = pd.read_csv("openaq_ground_sensor_data_complete.csv")
print(df.shape)
print(df.head())

(20974, 13)
         Date   PM2.5        PM10        NO         NO2  NH3      CO  \
0  2016-02-04  191.00  460.250000  30.75913  138.750000  NaN  4925.0   
1  2016-02-05  214.36  460.500000  30.75913   87.225000  NaN  2463.0   
2  2016-02-06  160.25  252.333333  30.75913   78.433333  NaN  1195.0   
3  2016-02-07  112.46  256.500000  30.75913   66.175000  NaN  1712.5   
4  2016-02-08  192.60  304.000000  30.75913   79.750000  NaN  1467.5   

         SO2         O3   City  Year  Month Seasons  
0  35.800000  13.265000  Delhi  2016      2  Winter  
1  35.775000  50.350000  Delhi  2016      2  Winter  
2  23.533333  31.233333  Delhi  2016      2  Winter  
3  23.375000  36.625000  Delhi  2016      2  Winter  
4  29.025000  47.575000  Delhi  2016      2  Winter  


In [16]:
df['City'].unique()

array(['Delhi', 'Mumbai', 'Chennai', 'Kolkata', 'Hyderabad', 'Pune',
       'Ahmedabad', 'Jaipur', 'Lucknow', 'Bangalore'], dtype=object)

## FEATURE ENGINEERING

In [3]:
BREAKPOINTS = {
    'PM2.5': [
        (0,   30,  0,   50),
        (31,  60,  51,  100),
        (61,  90,  101, 200),
        (91,  120, 201, 300),
        (121, 250, 301, 400),
        (251, 500, 401, 500),
    ],
    'PM10': [
        (0,   50,  0,   50),
        (51,  100, 51,  100),
        (101, 250, 101, 200),
        (251, 350, 201, 300),
        (351, 430, 301, 400),
        (431, 500, 401, 500),
    ],
    'NO2': [
        (0,   40,  0,   50),
        (41,  80,  51,  100),
        (81,  180, 101, 200),
        (181, 280, 201, 300),
        (281, 400, 301, 400),
        (401, 500, 401, 500),
    ],
    'SO2': [
        (0,   40,   0,   50),
        (41,  80,   51,  100),
        (81,  380,  101, 200),
        (381, 800,  201, 300),
        (801, 1600, 301, 400),
        (1601,2100, 401, 500),
    ],
    'CO': [                          
        (0,   1.0,  0,   50),
        (1.1, 2.0,  51,  100),
        (2.1, 10.0, 101, 200),
        (10.1,17.0, 201, 300),
        (17.1,34.0, 301, 400),
        (34.1,50.0, 401, 500),
    ],
    'O3': [                         
        (0,   50,  0,   50),
        (51,  100, 51,  100),
        (101, 168, 101, 200),
        (169, 208, 201, 300),
        (209, 748, 301, 400),
        (749, 900, 401, 500),
    ],
    'NH3': [                         
        (0,   200,  0,   50),
        (201, 400,  51,  100),
        (401, 800,  101, 200),
        (801, 1200, 201, 300),
        (1201,1800, 301, 400),
        (1801,2400, 401, 500),
    ],
}

def get_aqi_subindex(C, breakpoints):
    if C is None or C < 0:
        return None
    for (C_low, C_high, AQI_low, AQI_high) in breakpoints:
        if C_low <= C <= C_high:
           AQI = ((AQI_high - AQI_low) / (C_high - C_low)) * (C - C_low) + AQI_low
           return round(AQI)
    return None

def calculate_aqi(row):
    sub_indices = []

    for pollutant, bp_table in BREAKPOINTS.items():
        value = row[pollutant]
        sub_index = get_aqi_subindex(value, bp_table)
        if sub_index is not None:
            sub_indices.append(sub_index)

    if sub_indices:
        return max(sub_indices)
    return None


In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv('openaq_ground_sensor_data_complete.csv')
df['Date'] = pd.to_datetime(df['Date'].astype(str).str.split(' ').str[0])



df = df.sort_values(by=['City', 'Date']).reset_index(drop=True)

pollutants = ['PM2.5', 'PM10', 'NO', 'NO2', 'NH3', 'CO', 'SO2', 'O3']
df[pollutants] = df.groupby('City')[pollutants].transform(lambda x: x.ffill().bfill().fillna(0))


aqi_calc_df = df.copy()
aqi_calc_df['CO'] = aqi_calc_df['CO'] / 1000 
df['AQI_calculated'] = aqi_calc_df.apply(calculate_aqi, axis=1) 


df['DayOfWeek'] = df['Date'].dt.weekday

df['AQI_lag1']  = df.groupby('City')['AQI_calculated'].shift(1)
df['AQI_lag3']  = df.groupby('City')['AQI_calculated'].shift(3)
df['AQI_lag7']  = df.groupby('City')['AQI_calculated'].shift(7)
df['AQI_tomorrow'] = df.groupby('City')['AQI_calculated'].shift(-1) # The Target!

df['rolling_mean_7'] = df.groupby('City')['AQI_calculated'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

df['days_between_lag7'] = (df['Date'] - df.groupby('City')['Date'].shift(7)).dt.days
df = df[df['days_between_lag7'] <= 14].reset_index(drop=True) # Allow a max gap of 14 days

df = df.dropna(subset=['AQI_lag7', 'AQI_tomorrow']).reset_index(drop=True)


expected_cols = [
    'City', 'PM2.5', 'PM10', 'NO', 'NO2', 'NH3', 'CO', 'SO2', 'O3', 
    'Month', 'Seasons', 'AQI_calculated', 'AQI_lag1', 'AQI_lag3', 'AQI_lag7', 
    'DayOfWeek', 'rolling_mean_7', 'AQI_tomorrow'
]

final_ml_dataset = df[expected_cols]


final_ml_dataset.to_csv('final_xgboost_data_v3.csv', index=False)

In [10]:
df = pd.read_csv("final_xgboost_data_v3.csv")
df.head(5)

,City,PM2.5,PM10,NO,NO2,NH3,CO,SO2,O3,Month,Seasons,AQI_calculated,AQI_lag1,AQI_lag3,AQI_lag7,DayOfWeek,rolling_mean_7,AQI_tomorrow
0,Ahmedabad,154.0,181.5,15.598333,95.5,0.0,67500.0,61.5,16.0,3,Spring,326,341.0,471.0,497.0,3,392.428571,341.0
1,Ahmedabad,173.0,181.5,15.598333,104.0,0.0,93000.0,63.7,25.7,3,Spring,341,326.0,493.0,376.0,4,387.428571,407.0
2,Ahmedabad,267.0,181.5,15.598333,155.0,0.0,91000.0,105.0,14.5,3,Spring,407,341.0,341.0,389.0,5,390.000000,372.0
3,Ahmedabad,214.0,181.5,15.598333,131.0,0.0,60000.0,107.0,45.9,3,Spring,372,407.0,326.0,351.0,6,393.000000,390.0
4,Ahmedabad,237.0,181.5,15.598333,121.0,0.0,52000.0,76.3,77.3,3,Spring,390,372.0,341.0,471.0,0,381.428571,486.0


In [11]:
df.isnull().sum()

City              0
PM2.5             0
PM10              0
NO                0
NO2               0
NH3               0
CO                0
SO2               0
O3                0
Month             0
Seasons           0
AQI_calculated    0
AQI_lag1          0
AQI_lag3          0
AQI_lag7          0
DayOfWeek         0
rolling_mean_7    0
AQI_tomorrow      0
dtype: int64